In [ ]:
library(here)
library(maplet)
library(dplyr)
library(purrr)

# set repo path
repo <- here()
renv::activate(project = repo)

In [ ]:
# confirm the md5 checksum of the data
D <- D %>% maplet::mt_load_checksum(file = here('data', 'venous_metabolon.RDS'), checksum = '9cea47e7595183e825fa4d2c3b99eaa4')
D

# Preprocessing 

In [ ]:
# preprocess data
D2 <- D %>% 
    # remove 'Xenobiotics' and 'NA' metabolites
    mt_modify_filter_features(filter=SUPER_PATHWAY!="Xenobiotics" & !is.na(SUPER_PATHWAY)) %>% 
    mt_pre_filter_missingness(feat_max = 0.5) %>% # Filter out metabolites with more than 50% missing values
    mt_pre_norm_quot() %>% # Quotient normalize
    mt_pre_trans_log() %>%  # Log-transform (base 2)
    mt_pre_outlier_to_na() %>% 
    mt_pre_impute_knn()

In [ ]:
# save preprocessed data
saveRDS(D2, here('data', 'preprocessed_venous_metabolon.RDS'))

# Univarairate Regression

In [ ]:
# specify dirpath to store results 
dirpath <- here("outputs")

In [ ]:
# Define confounders as a string
confounders <- "age + SEX + bmi + f_wnowt"

## GLS 6`

In [ ]:
# specify variable of interest 
var_of_interest <- "RVGLOB6n"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D3 <- D2 %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# export results 
D3 <- D3 %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_univariate_test.xlsx"))) 

## FAC

In [ ]:
# specify variable of interest 
var_of_interest <- "RVFACn"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D3 <- D2 %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# export results 
D3 <- D3 %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_univariate_test.xlsx"))) 

## RVEF

In [ ]:
# specify variable of interest 
var_of_interest <- "mri_RVEF"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D3 <- D2 %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# export results 
D3 <- D3 %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_univariate_test.xlsx"))) 

# Secondary Analysis: Correct for Mean PA Pressure

In [ ]:
# remove samples that don't contain the PA_MP_MEAS_SPONT (mostly Healthy Controls)
D3 <- D2 %>% mt_modify_filter_samples(filter = !is.na(PA_MP_MEAS_SPONT))

# add PA_MP_MEAS_SPONT as another confounder
confounders <- "PA_MP_MEAS_SPONT + age + GENDER + bmi + f_wnowt"

## GLS 6`

In [ ]:
# specify variable of interest 
var_of_interest <- "RVGLOB6n"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D4 <- D3 %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met_mpa")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met_mpa"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met_mpa"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# export results 
D4 <- D4 %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_mPA_corrected_univariate_test.xlsx"))) 

## FAC

In [ ]:
# specify variable of interest 
var_of_interest <- "RVFACn"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D4 <- D3 %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met_mpa")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met_mpa"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met_mpa"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# export results 
D4 <- D4 %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_mPA_corrected_univariate_test.xlsx"))) 

## RVEF

In [ ]:
# specify variable of interest 
var_of_interest <- "mri_RVEF"

# Construct the lm formula
full_formula <- as.formula(paste("~", var_of_interest, "+", confounders))

# run univariate tests
D4 <- D3 %>%
  # Linear model with dynamically created formula
  mt_stats_univ_lm(formula = full_formula, stat_name = paste(var_of_interest, "met_mpa")) %>%
  # Add multiple testing correction
  mt_post_multtest(stat_name = paste(var_of_interest, "met_mpa"), method = "fdr") %>%
  # Add stats logging
  mt_reporting_stats(stat_name = paste(var_of_interest, "met_mpa"), stat_filter = p.adj < 0.05) %>% 
  {.}

In [ ]:
# export results 
D4 <- D4 %>% mt_write_stats(file = here("outputs", paste0(var_of_interest, "_mPA_corrected_univariate_test.xlsx"))) 